# C8-embeddings — Practice p17 — Solution

In [ ]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

In [ ]:
kv = gensim.downloader.load("glove-wiki-gigaword-100")

LISTS = [
    ["snow", "rain", "fog", "thunder", "cello"],
    ["guitar", "piano", "flute", "trumpet", "salmon"],
    ["bread", "cheese", "honey", "garlic", "falcon"],
    ["otter", "heron", "badger", "salmon", "trumpet"],
]


def odd_one_out(kv, words):
    V = np.asarray(kv[words], dtype=np.float64)
    W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
    S = W @ W.T
    # Subtracting one removes each unit row's self-similarity from its row sum.
    means = (S.sum(axis=1) - 1.0) / (len(words) - 1)
    return words[int(np.argmin(means))]


odd_words = []
odd_means = []
for words in LISTS:
    intruder = odd_one_out(kv, words)
    V_group = np.asarray(kv[words], dtype=np.float64)
    W_group = V_group / np.sqrt((V_group * V_group).sum(axis=1, keepdims=True))
    S_group = W_group @ W_group.T
    group_means = (S_group.sum(axis=1) - 1.0) / (len(words) - 1)
    odd_words.append(intruder)
    odd_means.append(round(float(group_means[words.index(intruder)]), 4))

The geometry identifies the thematically mismatched items as `cello`, `salmon`, `falcon`, and `trumpet`. Their mean similarities to the other four words are respectively 0.0379, 0.1267, -0.0104, and 0.1699.

### Answer check

In [ ]:
assert odd_words == ["cello", "salmon", "falcon", "trumpet"]
assert odd_means == [0.0379, 0.1267, -0.0104, 0.1699]
assert all(isinstance(value, float) for value in odd_means)